In [1]:
#!pip uninstall -y transformers accelerate peft bitsandbytes trl
#!pip install -q -U bitsandbytes accelerate peft transformers trl datasets
#!pip install faiss-cpu

In [2]:
import os
import json
import torch
import numpy as np
from typing import List, Dict, Tuple
from dataclasses import dataclass
import pickle
from tqdm import tqdm

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from sentence_transformers import SentenceTransformer
import faiss

import wandb
from huggingface_hub import notebook_login

In [3]:
"""Install required packages and login to Hugging Face"""

print("🔧 Setting up environment...")

# Check for GPU
if torch.cuda.is_available():
    print(f"✓ GPU detected: {torch.cuda.get_device_name(0)}")
    print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️  No GPU detected. Training will be very slow!")

# Login to Hugging Face (needed for Llama 3)
print("\n🔐 Please login to Hugging Face:")
print("   You need access to Llama 3 model (request at: https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct)")
notebook_login()

print("✓ Environment setup complete\n")

🔧 Setting up environment...
✓ GPU detected: NVIDIA GeForce RTX 5090 Laptop GPU
  Memory: 25.65 GB

🔐 Please login to Hugging Face:
   You need access to Llama 3 model (request at: https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct)
✓ Environment setup complete



In [4]:
"""
LangGraph Joke Generation Pipeline - Orchestrator
Imports and orchestrates existing implementations
"""
from typing import TypedDict, Literal, List, Dict
from langgraph.graph import StateGraph, END
from pathlib import Path

import sys

# ✅ set project root = folder c:\Users\Anwender\humor-project
PROJECT_ROOT = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()

# if you run from a notebook inside some subfolder, walk up until we find "src"
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))         # so "import src...." works
sys.path.insert(0, str(PROJECT_ROOT / "src")) # so "import evaluation...." works if you want
print("Using PROJECT_ROOT:", PROJECT_ROOT)
# Import your existing implementations
from using_llama_with_rag import RAGHumorGenerator, RAGConfig
from gpt_joke_generation import GPT4JokeGenerator
from src.evaluation.gpt_oss_evaluator import GPTOSSHumorScorer

Using PROJECT_ROOT: c:\Users\Anwender\humor-project


In [5]:
# ============================================================================
# LANGGRAPH STATE DEFINITION
# ============================================================================

class JokeGenerationState(TypedDict):
    """State for the joke generation pipeline"""
    # Input
    word1: str
    word2: str

    # Generation tracking
    llama_jokes: List[str]
    gpt_jokes: List[str]
    candidate_jokes: List[str]

    # Scoring tracking
    scores: List[float]

    # Best result tracking
    best_joke: str
    best_score: float

    # Iteration tracking
    iteration_count: int
    max_iterations: int

    # History for debugging
    history: List[Dict]


# ============================================================================
# NODE FUNCTIONS
# ============================================================================

def generate_jokes_llama(state: JokeGenerationState, llama_gen: RAGHumorGenerator) -> Dict:
    """Generate jokes using Llama model"""
    word1 = state["word1"]
    word2 = state["word2"]
    iteration = state.get("iteration_count", 0)

    print(f"\n[Iteration {iteration + 1}] Generating jokes with Llama ...")

    result_with_RAG = llama_gen.generate_with_rag(
        word1=word1,
        word2=word2,
        use_rag=True,  # with RAG
        num_candidates=5
    )

    result_without_RAG = llama_gen.generate_with_rag(
        word1=word1,
        word2=word2,
        use_rag=False,  # No RAG - just fine-tuned model
        num_candidates=5
    )

    jokes = result_with_RAG["generated_jokes"] + result_without_RAG["generated_jokes"]

    print('result_with_RAG jokes:')
    print(result_with_RAG["generated_jokes"])

    print('result_without_RAG jokes:')
    print(result_without_RAG["generated_jokes"])

    print(f"   Generated {len(jokes)} Llama jokes")

    return {"llama_jokes": jokes}


def generate_jokes_gpt(state: JokeGenerationState, gpt_gen: GPT4JokeGenerator) -> Dict:
    """Generate jokes using GPT-OSS"""
    word1 = state["word1"]
    word2 = state["word2"]
    iteration = state.get("iteration_count", 0)

    print(f"\n[Iteration {iteration + 1}] Generating jokes with GPT-OSS...")

    jokes = gpt_gen.generate_jokes(word1, word2, num_jokes=5, verbose=False)

    print(f"   Generated {len(jokes)} GPT-OSS jokes")

    return {"gpt_jokes": jokes}


def combine_jokes(state: JokeGenerationState) -> JokeGenerationState:
    """Combine jokes from both generators"""
    llama_jokes = state.get("llama_jokes", [])
    gpt_jokes = state.get("gpt_jokes", [])

    all_jokes = llama_jokes + gpt_jokes

    print(f"\n✅ Combined {len(all_jokes)} total jokes ({len(llama_jokes)} Llama + {len(gpt_jokes)} GPT)")

    return {
        **state,
        "candidate_jokes": all_jokes
    }


def score_jokes_gpt_oss(state: JokeGenerationState, scorer: GPTOSSHumorScorer) -> JokeGenerationState:
    """Score all candidate jokes using GPT-OSS scorer"""
    jokes = state["candidate_jokes"]
    word1 = state["word1"]
    word2 = state["word2"]
    iteration = state.get("iteration_count", 0)

    print(f"\n📊 [Iteration {iteration + 1}] Scoring {len(jokes)} jokes with GPT-OSS...")

    scores = scorer.score_jokes(jokes, word1, word2, verbose=False)

    print(scores)
    for joke in jokes:
      print(joke)

    # Find best joke
    if scores:
        best_idx = scores.index(max(scores))
        best_joke = jokes[best_idx]
        best_score = scores[best_idx]
    else:
        best_joke = ""
        best_score = 0.0

    print(f"   Best score: {best_score:.2f}/10")
    print(f"   Average score: {sum(scores)/len(scores):.2f}/10")
    print(f"   Score range: {min(scores):.2f} - {max(scores):.2f}")

    return {
        **state,
        "scores": scores,
        "best_joke": best_joke,
        "best_score": best_score
    }


def check_score_threshold(state: JokeGenerationState) -> Literal["refine", "output"]:
    """Decision node: check if score meets threshold"""
    score = state["best_score"]
    iteration = state.get("iteration_count", 0)
    max_iter = state.get("max_iterations", 3)

    print(f"\n🎯 Decision: Score={score:.2f}, Iteration={iteration + 1}/{max_iter}")

    if score >= 8.0:
        print(f"   ✓ Score threshold met! (≥8.0)")
        return "output"
    elif iteration >= max_iter - 1:
        print(f"   ⚠️  Max iterations reached, outputting best result")
        return "output"
    else:
        print(f"   🔄 Score below threshold, refining...")
        return "refine"


def increment_iteration(state: JokeGenerationState) -> JokeGenerationState:
    """Increment iteration counter and save history"""
    current_iter = state.get("iteration_count", 0)
    history = state.get("history", [])

    # Save this iteration's results
    history.append({
        "iteration": current_iter + 1,
        "best_score": state["best_score"],
        "best_joke": state["best_joke"],
        "num_candidates": len(state.get("candidate_jokes", [])),
        "avg_score": sum(state.get("scores", [0])) / max(len(state.get("scores", [1])), 1)
    })

    return {
        **state,
        "iteration_count": current_iter + 1,
        "history": history
    }


def output_result(state: JokeGenerationState) -> JokeGenerationState:
    """Final output node"""
    print(f"\n{'='*80}")
    print(f"🎭 FINAL RESULT")
    print(f"{'='*80}")
    print(f"\nWord Pair: '{state['word1']}' + '{state['word2']}'")
    print(f"Final Score: {state['best_score']:.2f}/10")
    print(f"Total Iterations: {state.get('iteration_count', 0) + 1}")
    print(f"\nSelected Joke:")
    print(f"  {state['best_joke']}")

    # Show iteration progress
    if state.get("history"):
        print(f"\n📈 Iteration History:")
        for h in state["history"]:
            print(f"   Iteration {h['iteration']}: Best={h['best_score']:.2f}, Avg={h['avg_score']:.2f}, Candidates={h['num_candidates']}")

    print(f"\n{'='*80}\n")

    return state


# ============================================================================
# GRAPH BUILDER
# ============================================================================

def build_joke_generation_graph(
    llama_generator: RAGHumorGenerator,
    gpt_generator: GPT4JokeGenerator,
    scorer: GPTOSSHumorScorer
):
    """
    Build the LangGraph for joke generation pipeline.

    Args:
        llama_generator: RAGHumorGenerator
        gpt_generator: GPT4JokeGenerator
        scorer: GPTOSSHumorScorer

    Returns:
        Compiled LangGraph
    """
    workflow = StateGraph(JokeGenerationState)

    # Add nodes with bound generators/scorer
    workflow.add_node(
        "generate_llama",
        lambda state: generate_jokes_llama(state, llama_generator)
    )
    workflow.add_node(
        "generate_gpt",
        lambda state: generate_jokes_gpt(state, gpt_generator)
    )
    workflow.add_node("combine_jokes", combine_jokes)
    workflow.add_node(
        "score_jokes",
        lambda state: score_jokes_gpt_oss(state, scorer)
    )
    workflow.add_node("increment_iteration", increment_iteration)
    workflow.add_node("output", output_result)

    # Set entry point - start both generators in parallel
    workflow.set_entry_point("generate_llama")
    workflow.set_entry_point("generate_gpt")

    # Add edges - both generators feed into combine
    workflow.add_edge("generate_llama", "combine_jokes")
    workflow.add_edge("generate_gpt", "combine_jokes")

    # After combining, score the jokes
    workflow.add_edge("combine_jokes", "score_jokes")

    # Decision point: refine or output
    workflow.add_conditional_edges(
        "score_jokes",
        check_score_threshold,
        {
            "refine": "increment_iteration",
            "output": "output"
        }
    )

    # Refinement loop - go back to both generators
    workflow.add_edge("increment_iteration", "generate_llama")
    workflow.add_edge("increment_iteration", "generate_gpt")

    # End the workflow
    workflow.add_edge("output", END)

    return workflow.compile()

In [6]:
print("="*80)
print("LANGGRAPH JOKE GENERATION PIPELINE - ORCHESTRATOR")
print("="*80)
print()

# Configuration
NVIDIA_API_KEY = "REDACTED_CREDENTIAL"

config = RAGConfig()
config.TEMPERATURE = 0.65
config.TOP_K = 50

# Initialize components
print("🚀 Initializing components...\n")

# 1. Initialize Llama Generator
try:
    llama_gen = RAGHumorGenerator(config)
    llama_gen.load_model()
    llama_gen.load_rag_database(build_if_missing=True)

    llama_available = True
except Exception as e:
    print(f"⚠️  Could not load Llama model: {e}")
    print("   You can still run GPT-only mode with joke_generation_langgraph_simple.py\n")
    sys.exit(1)

# 2. Initialize GPT-OSS Generator
gpt_gen = GPT4JokeGenerator(
    api_key=NVIDIA_API_KEY,
    temperature=0.8,
    max_retries=1
)
print("✓ GPT-OSS generator initialized\n")

# 3. Initialize GPT-OSS Scorer
scorer = GPTOSSHumorScorer(
    api_key=NVIDIA_API_KEY,
    temperature=0.4,
    max_retries=2
)
print("✓ GPT-OSS scorer initialized\n")

LANGGRAPH JOKE GENERATION PIPELINE - ORCHESTRATOR

🚀 Initializing components...

🤖 Loading Fine-tuned Model...
  ✓ Tokenizer loaded


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

  ✓ Base model loaded
  ✓ LoRA weights loaded
  ✓ Model ready on cuda

📂 Loading RAG Database...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Loaded database with 5000 jokes

✓ GPT-OSS generator initialized

✓ GPT-OSS scorer initialized



In [8]:
# Build the graph
graph = build_joke_generation_graph(llama_gen, gpt_gen, scorer)
print("✓ LangGraph compiled (Llama + GPT-OSS mode)\n")

# Test word pairs
test_pairs = [
    ("banana", "satellite"),
    ("angry", "teacup"),
    ("cosmic", "toaster"),
    ("sleepy", "volcano"),
    ("neon", "hamster"),
    ("awkward", "penguin"),
    ("electric", "spaghetti"),
    ("dramatic", "cactus"),
    ("confused", "astronaut"),
    ("rubber", "tornado"),
    ("invisible", "taco"),
    # ("sarcastic", "dolphin"),
    # ("haunted", "calculator"),
    # ("spicy", "cloud"),
    # ("polite", "chainsaw"),
    # ("disco", "librarian"),
    # ("lonely", "robot"),
    # ("caffeinated", "squirrel"),
    # ("confused", "mailbox")
]

# Run the pipeline for each word pair
for word1, word2 in test_pairs:
    print("\n" + "="*80)
    print(f"PROCESSING: '{word1}' + '{word2}'")
    print("="*80)

    # Initial state
    initial_state = {
        "word1": word1,
        "word2": word2,
        "llama_jokes": [],
        "gpt_jokes": [],
        "candidate_jokes": [],
        "scores": [],
        "best_joke": "",
        "best_score": 0.0,
        "iteration_count": 0,
        "max_iterations": 1,
        "history": []
    }

    try:
        # Run the graph
        result = graph.invoke(initial_state)

        # Additional result summary
        print(f"\n{'🎨'*40}")
        print("SUMMARY")
        print(f"{'🎨'*40}")
        print(f"Total Jokes Generated: {sum(h['num_candidates'] for h in result.get('history', []))} + {len(result.get('candidate_jokes', []))}")
        print(f"Success: {'✓' if result['best_score'] >= 8.0 else '✗'} (threshold: 8.0)")

    except Exception as e:
        print(f"\n❌ Error processing word pair: {e}")
        import traceback
        traceback.print_exc()

print("\n" + "="*80)
print("✅ Pipeline execution complete!")
print("="*80)

✓ LangGraph compiled (Llama + GPT-OSS mode)


PROCESSING: 'banana' + 'satellite'

[Iteration 1] Generating jokes with GPT-OSS...

[Iteration 1] Generating jokes with Llama ...
   Generated 5 GPT-OSS jokes
result_with_RAG jokes:
['What do you call a banana with a satellite in it? A fruit with a nut.', "What do you call a banana that's been to the moon? A satellite fruit.", 'How do you get a banana into space? Put it in a satellite.', 'If a girl says she’s leaving because you’re boring, you should take it as a compliment. It means she thinks you’re a banana and she’s a fucking satellite.', 'What do you call a banana that is also a satellite? A yellow brick road.']
result_without_RAG jokes:
["What do you call a banana that's been hit by a satellite? A peanut.", 'What do you call a banana with a GPS device? A satellite.', "What do you call a yellow fruit that's also a satellite? A banana-satellite.", "What do you call a banana that's been shot into space? A satellite fruit.", "What do you 